In [1]:
# Install required packages (run once)
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Module 1: What Is Memory?

In this module you'll explore **memory** — what it is, how agents use it, and why the default kind isn't enough.

## What You'll Learn
1. Every agent already has **session memory** (conversation context)
2. Session memory is **transient** — lost when the session ends or you restart
3. Why we need **persistent memory** backed by external storage

## Scenario: Corporate Travel Assistant
We're building a travel assistant for Contoso Corp employees.
We'll use the **Microsoft Agent Framework** as our implementation tool throughout.

---
## Understanding Memory

### Every Agent Has Session Memory

All conversational agents maintain **session memory** — the running history of messages in a conversation. This isn't a special feature; it's how multi-turn conversations work. The agent accumulates messages:

```
messages = [
    {"role": "user",      "content": "Hi, I'm Sarah"},
    {"role": "assistant", "content": "Hello Sarah!"},
    {"role": "user",      "content": "What's my name?"},
]
→ The model sees the full history and knows "Sarah"
```

Whether you manage this array yourself or use a framework, the result is the same — the agent remembers what was said **within the current session**.

> **Important:** In the Microsoft Agent Framework, each call to `agent.run()` is stateless by default. To maintain conversation context across multiple turns, you must create an `AgentSession` and pass it to every `run()` call. Without a session, each `run()` starts a completely fresh conversation.

### The Real Question: What Happens When the Session Ends?

Session memory is **transient**. When the session ends — the process stops, the app restarts, the user comes back tomorrow — that conversation history is gone.

This is the central problem of the module series:

| Memory Type | Lifespan | Example |
|-------------|----------|---------|
| **Session (transient)** | Current conversation only | "I'm going to NYC next month" |
| **Persistent** | Survives across sessions | "Sarah prefers window seats" (stored externally) |

In this module we'll see session memory in action and experience its limitation. In later modules, we'll add persistent memory.

---
## Setup

We'll use the Microsoft Agent Framework to create our travel agent. The framework handles session memory automatically — we just focus on the conversations.

In [2]:
import os
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# Enable async in Jupyter
nest_asyncio.apply()

# Load environment variables
load_dotenv("../../.env", override=True)

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ.get("FOUNDRY_MODEL_DEPLOYMENT_NAME", "gpt-5-mini")

print(f"Project: {PROJECT_ENDPOINT}")
print(f"Model: {MODEL_DEPLOYMENT}")

Project: https://npmsfoundrysc.services.ai.azure.com/api/projects/proj-default
Model: gpt-5.4-mini


In [3]:
import os
from azure.identity import AzureCliCredential
from agent_framework import Agent, AgentSession
from agent_framework.foundry import FoundryChatClient

credential = AzureCliCredential()

client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT,
    credential=credential,
)
print(f"Client initialized for: {PROJECT_ENDPOINT}")

Client initialized for: https://npmsfoundrysc.services.ai.azure.com/api/projects/proj-default


---
## Part 1: A Single Conversation Turn

Let's start simple — one message, one response. No memory demonstrated yet.

In [4]:
SYSTEM_INSTRUCTIONS = """You are a corporate travel assistant for Contoso Corp. 
Help employees book business travel including flights and hotels.
Be concise and helpful."""

async def single_turn():
    agent = Agent(
        client=client,
        name="TravelAssistant",
        instructions=SYSTEM_INSTRUCTIONS,
    )
    response = await agent.run("Hi, I'm Sarah Chen from Engineering.")
    print(f"Agent: {response.text}")

asyncio.run(single_turn())

Agent: Hi Sarah — nice to meet you. I can help with business travel bookings for Contoso Corp.

What would you like to arrange: flight, hotel, or both?


### One Turn Down — But Where's the Memory?

A single exchange doesn't show memory at work. Memory matters when the agent needs to **recall earlier parts of the conversation**.

Let's have a multi-turn conversation and see session memory in action.

---
## Part 2: Session Memory in Action

Within a single session, the agent maintains conversation context across multiple turns.

To enable this, we create an `AgentSession` and pass it via `session=session` to each `agent.run()` call. This tells the framework to accumulate the conversation history across turns:

```
┌─────────────────────────────────────────────────────────┐
│                   Agent Session                          │
│                                                          │
│  Turn 1: "I'm Sarah, going to NYC"                      │
│  Turn 2: "What about hotels?"         ← Knows NYC       │
│  Turn 3: "Book the Marriott"          ← Knows NYC       │
│                                                          │
│  Session Memory: [turn1, turn2, turn3]                  │
└─────────────────────────────────────────────────────────┘
```

This is **session memory** — every agent has it, but it must be explicitly wired up with an `AgentSession`.

In [5]:
async def demo_session_memory():
    """Multi-turn conversation — session memory keeps context."""

    agent = Agent(
        client=client,
        name="TravelAssistant",
        instructions=SYSTEM_INSTRUCTIONS,
    )

    session = AgentSession()

    # Turn 1: Introduce ourselves
    print("User: Hi, I'm Sarah Chen from Engineering.")
    response = await agent.run("Hi, I'm Sarah Chen from Engineering.", session=session)
    print(f"Agent: {response.text}\n")

    # Turn 2: Add context — agent should remember Turn 1
    print("User: I need to travel to New York next month.")
    response = await agent.run("I need to travel to New York next month.", session=session)
    print(f"Agent: {response.text}\n")

    # Turn 3: Test recall — does the agent remember name + destination?
    print("User: What's my name and where am I going?")
    response = await agent.run("What's my name and where am I going?", session=session)
    print(f"Agent: {response.text}")

asyncio.run(demo_session_memory())

User: Hi, I'm Sarah Chen from Engineering.
Agent: Hi Sarah — nice to meet you. How can I help with your business travel today?

User: I need to travel to New York next month.
Agent: Absolutely — I can help with that.

To get started, please share:
- Your departure city
- Preferred travel dates next month
- Whether you need just a flight, just a hotel, or both
- Any preferences or requirements, like nonstop flights, hotel budget, or loyalty programs

User: What's my name and where am I going?
Agent: Your name is Sarah Chen, and you're going to New York next month.


### Session Memory Works

The agent remembers:
- ✅ Your name (Sarah Chen)
- ✅ Your destination (New York)

The key is the `AgentSession` — it accumulates the conversation history so the agent sees all prior turns. Without it, each `agent.run()` call would be isolated with no context from previous turns.

---
## Part 3: Memory Lifespans — Transient vs Persistent

Session memory works great **within** a session. But what are its boundaries?

### Transient Memory (What We Have Now)

| Characteristic | Description |
|----------------|-------------|
| **Storage** | In-memory (conversation history) |
| **Lifespan** | Current session only |
| **Lost when** | Session ends, app restarts, kernel restarts |
| **Use case** | Conversation context |

### Persistent Memory (What We'll Build)

| Characteristic | Description |
|----------------|-------------|
| **Storage** | External — databases, graph stores, search indexes |
| **Lifespan** | Until explicitly deleted |
| **Survives** | Restarts, new sessions, different users |
| **Use case** | Preferences, past trips, policies, knowledge |

In [6]:
async def demo_transient_limitation():
    """Show that session memory doesn't survive across sessions."""

    agent = Agent(
        client=client,
        name="TravelAssistant",
        instructions=SYSTEM_INSTRUCTIONS,
    )

    # Session 1: Session memory works within the session
    print("--- Session 1: Memory works within the session ---")
    session = AgentSession()
    await agent.run("I'm Sarah and I prefer window seats.", session=session)
    response = await agent.run("What seat do I prefer?", session=session)  # ← Remembers!
    print(f"Agent: {response.text}")

    # Session 2: New session — no memory of Session 1
    print("\n--- Session 2: New session = fresh start ---")
    response = await agent.run("What seat do I prefer?")  # ← Doesn't know (no session)
    print(f"Agent: {response.text}")

asyncio.run(demo_transient_limitation())

--- Session 1: Memory works within the session ---
Agent: You prefer **window seats**.

--- Session 2: New session = fresh start ---
Agent: I don’t have your seat preference on file yet. If you want, tell me your preferred seat type and I’ll remember it for future bookings.


### The Transient Limitation

**Session 1** remembers the window seat preference — session memory works!

**Session 2** has no idea — it's a new session with a blank slate.

> **This is the core problem**: session memory is transient. Anything the agent learned is lost when the session ends.

### 🧪 Try It: Experience This Yourself

1. Run the multi-turn demo in Part 2 — the agent remembers your name
2. **Restart the kernel** (Kernel → Restart or ⟳)
3. Re-run just the last turn ("What's my name?") — the agent won't know

The kernel restart simulates what happens in production when your app restarts, deploys, or the user starts a new chat session.

---
## Summary

### What We Learned About Memory

| Concept | Key Insight |
|---------|-------------|
| **Session Memory** | Every agent has it — conversation context within a session |
| **Transient** | Session memory is lost when the session ends or restarts |
| **Persistent** | Requires external storage to survive across sessions |

### The Problem We'll Solve

```
Session 1: "I prefer window seats"  →  Agent knows ✅
                    ↓ session ends
Session 2: "What seat do I prefer?"  →  Agent doesn't know ❌
```

In the next modules, we'll add **persistent memory** so the agent remembers across sessions:

| Module | Memory Type | What It Remembers |
|--------|-------------|-------------------|
| **Module 2** | Episodic Memory | Past events and experiences |
| **Module 3** | Semantic Memory | Facts, preferences, relationships |
| **Module 4** | Procedural Memory | Policies and procedures |